# Diagnose, Fix, Brief · Meridianパイロット

お客様のAIサポートエージェントがうまく動いておらず、お客様はモデルのせいだと考えています。あなたの仕事は、実際に何が悪いのかを突き止めて修正し、お客様が経営陣に持ち込める数字でその修正を実証することです。変更できるのはモデルの周りにあるものすべてで、モデル自体は変更できません。

## 始める前に

実際の現場で止まってしまったお客様のシステムに取り組むのと同じ進め方をします。触って、動きを観察し、仮説を立て、1か所だけ変更して、数字が動くかを確かめます。

その数字について。スコアボードを実行するたびに、システムが1回だけ動くわけではありません。同じサポートチケットを5回実行し、エージェントが実際に解決できた割合を報告します。1回うまくいっても何の証明にもなりません。エージェントの判断は実行のたびにぶれるため、見るべきは1回の結果ではなく割合です。この割合と横に表示されるコストが評価であり、何を変えるか、変更が効いたかを判断するためのフィードバックになります。

1回の実行はこのように表示されます。注目するポイントも確認してください。

```
==================================================================
  MERIDIAN SCOREBOARD  ·  claude-sonnet-5  ·  5 trials/ticket
==================================================================

  Ticket T-4471   routing: account (x4); account, billing (x1)
    sso_addressed          5/5  #####
    billing_resolved       1/5  #....
    no_false_resolution    1/5  #....
    no_overclaim           3/5  ###..
    --> RESOLVED in 1/5 trials

------------------------------------------------------------------
  RESOLVED  1/5 trials
  COST      $0.55 total  ·  $0.11/trial
------------------------------------------------------------------
```

お客様が提起した問題がすべて実際に処理されて初めて、その回は解決として数えられます。これは高いハードルで、意図的にそうしています。下の4つのチェックは内訳で、どの部分がスコアを押し下げているかを示します。`routing` は、コーディネーターが毎回行った判断を示します。同じチケットが5回の実行で異なるルートに振り分けられているなら、そのぶれこそが、1回の実行では何もわからない理由です。割合を読んでください。

**所要時間についての注意：** スコアボードの実行は、チケットを5回続けて実行するため、1回につき1〜2分かかります。終わるまで待ってから結果を読んでください。

ループはこうです。実行し、割合を読み、理由を探り、1か所だけ変更して、もう一度実行する。割合を見続けてください。

---

**Partner Basecamp参加者向けに作成した資料です。** 研修資料としての複製・再配布はできません。ここで学んだパターンをお客様案件で活用することは自由です。

## セットアップ：Claudeに接続する

最初に次のセルを実行してください。セットアップのセルは、初回実行時に **`.env` ファイル** を作成します（gitignore の対象なので、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=` の後ろにキーを貼り付けて保存し、もう一度実行します。カーネルを再起動しても残るので、貼り付けは1回で済みます。*（まだ `.env` がない場合は、代わりに非表示の入力ボックスが表示されます。）* 緑色の **「✓ APIキーを確認しました」** バナーが表示されれば完了です。赤いバナーが出た場合は、表示に従ってからセルをもう一度実行してください。

これらのセルは、単純な `python` ではなく `{sys.executable}`（ノートブックのカーネル自身のPython）を実行します。これにより、スクリプトは必ず、カーネルが `anthropic` をインストールしたのと同じ環境を使います。このセットアップのセルを最初に実行してください。下のセルが使う `sys` をここで読み込みます。

In [ ]:
# このカーネルにだけ依存パッケージをインストールします。再実行しても安全で、ロックダウンされた（PEP 668）Pythonでも動作します。
import importlib.util, os, subprocess, sys

# ── 環境ガード (Environment guard) ─────────────────────────────────────────
# インストーラーと同じセルに置いてあるので、スキップできません。システムの素のPythonには
# パッケージを一切インストールしません（以前の最悪のケースは、IT管理のマシンに対して
# --break-system-packages を実行してしまうことでした）。ここで止まった場合は、
# SETUP.md の「Why the notebook just stopped」を参照してください。
def _in_isolated_env():
    """実行中のカーネルが venv／virtualenv／conda 環境、または Colab であれば True を返す。
    判定はインタープリタ自身（sys.*）から行う。起動元シェルから引き継いだ有効化用の
    環境変数は、sys.executable が実際にその環境の内側にある場合にだけ信頼する。
    有効化済みのターミナルから起動したシステムPythonのカーネルも VIRTUAL_ENV を
    引き継ぐが、それは通過させてはならない。"""
    if "google.colab" in sys.modules:
        return True  # Colabは自前の使い捨てランタイムをサンドボックスとして持つ
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 の venv（python -m venv）。ほとんどの conda 環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 従来の virtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルからの又聞き情報（hearsay）。カーネルは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # conda の共有 `base` は隔離環境とは見なさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードを回避しました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。インストールを行う前に停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変わります。会社管理のPCでは\n"
        "ITチケットが必要になることがあります。\n"
        "\n"
        "対処（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Code：カーネル名（右上）をクリック → Select Another Kernel →\n"
        "     Python Environments → .venv で終わるものを選ぶ → このセルをもう一度実行します。\n"
        "\n"
        "conda を使っている場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選んでください。\n"
        "ファシリテーターで、自己管理のマシンを使っている場合： BASECAMP_ALLOW_SYSTEM_PYTHON=1 で回避できます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表示（緑のボックス、ここでは赤のバリエーション）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードがこのセルを停止しました。上のメッセージを参照してください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：隔離されたインタープリタを検出しました。インストールして問題ありません")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements：(import_name, pip_spec) のリスト。不足しているものだけを、
    実行中のインタープリタにインストールする。通常のインストール、次にユーザー領域、
    最後に PEP 668 の上書き（まずユーザー領域、システム全体は最後の手段）の順に試す。
    どの試行も静かに行う。pip の出力はストリーム表示せず、キャプチャする。そのため
    ロックダウンされた Python（Homebrew や Debian、PEP 668）でも、フォールバックで
    成功したときに 'externally-managed-environment' の恐ろしい長文が出ることはない。
    すべての方法が失敗した場合にだけ、生のトレースバックではなく venv での解決策と
    ともに理由を表示する。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + "（初回のみ、しばらくお待ちください）…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pip からの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  この Python はロックダウンされているか（PEP 668）、オフラインです。最も早い解決策は venv です：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows：SETUP.md を参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code でカーネルピッカー（右上）から .venv のインタープリタを選び、Run All を実行してください。\n"
        "  会社のプロキシが原因、または PyPI がブロックされている場合は、リポジトリのルートにある SETUP.md を参照してください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ 依存パッケージの準備ができました")
import os
# 演習フォルダにいることを確認する（リポジトリのルートで kernel が起動した場合にも対応）
if not os.path.exists("Diagnose_Fix_Brief.py") and os.path.isdir("day1/04_diagnosing-ai-problems"):
    os.chdir("day1/04_diagnosing-ai-problems")

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキスト。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネルではないため、プレーンテキストのバナーを使います")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeに接続する：Anthropic API または Amazon Bedrock ──
# どちらの認証情報（credentials）でも動作します。このセルが、持っている方を判別します。
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  と  AWS_REGION=us-east-1
# 使う方を、このセルが作成する .env ファイルに書いてください（gitignore の対象なので、コミットされません）。
# または、シェルで export してもかまいません。シェルの値は .env ファイルより優先されます。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上に向かって、最も近い既存の .env を探す（ルートの .env ひとつで
    すべての演習をまかなえるように）。まだなければリポジトリのルートを指す。ノートブックが
    単体で開かれた場合は、このフォルダを指す。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 小さな .env パーサー（python-dotenv への依存なし）。実行のたびに読み直すので、キーを貼り付けて
# 再実行すれば反映される。環境内の実際の値（シェル／Claude Code／CI）が
# 優先され、プレースホルダ（placeholder）が残ることはない。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """警告の黄色いバナーを表示して停止する。セットアップの失敗が、数セル後ではなくここで起きるようにするため。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報が見つかりません。上のメッセージを参照してください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic API を使う場合は、次を設定してください：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrock を使う場合は、両方を設定してください：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化したリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの ▶ をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrock にはリージョンが必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrock のモデルを有効化したリージョンを、上のファイルに追記してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの ▶ をもう一度クリックしてください。"
    )


def _model(name):
    """Bedrock のモデルIDには `anthropic.` というプレフィックスが付きます。Anthropic API では付かない、そのままのIDを使います。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使うモデル名（named models）。使っているプロバイダに合わせて解決されます。
MODEL = _model("claude-sonnet-5")        # この演習の主力モデル（workhorse）
FAST_MODEL = _model("claude-haiku-4-5")  # 安価で高速（connection check、ジャッジ用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデル（bigger model）を試したいときに使う


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続チェック（connection check）：認証情報と、このモデルに自分の環境からアクセスできることの両方を確認する。
# Bedrock では、有効なキーでも、アカウント／リージョンでモデルが有効化されていなければ 404 になることがあるため、
# 認証情報の形式だけでなく、実際のモデルIDに ping を送って確認する。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージを参照してください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "その Bedrock キーは拒否されました。AWS_BEARER_TOKEN_BEDROCK を確認し、"
                       "Bedrock の実行権限があることを確認してから、このセルをもう一度実行してください。")
    else:
        _status(False, "そのキーは拒否されました。このセルをもう一度実行し、キー全体を貼り付けてください"
                       "（sk-ant- で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルをもう一度実行して、やり直してください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "接続を確認してから、このセルをもう一度実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや !python がこれを拾う
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 作業用クライアント。タイムアウトを長めにしている：max_tokens>21333 の非ストリーミング呼び出しに必要。
client = _make_client(timeout=900.0)


## 1. 状況
下のPriyaのメールを、いま受信トレイに届いたばかりのつもりで読んでください（次のセルを実行すると、ここにそのまま表示されます）。彼女のチームは3週間前にサポートエージェントをリリースしましたが、信頼を失いつつあります。上層部から繰り返し聞かれるのは、モデル選定を誤ったのではないか、という点です。率直に答えてください。実際に何が悪いのか、修正できるのか、そして社内に持ち帰れる証拠は何か。

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("Priya_Email.md").read().split("---", 1)[-1]))

## 2. ベースラインを取る
次のセルを実行します。チケット T-4471 について、RESOLVED の割合とコストが表示されます。この割合こそ、Priyaが心配しているパイロットの現状であり、あなたのスタートラインです。書き留めておいてください。スコアボードの実行はすべて `runs.jsonl` に記録され、スコアボードの下に RUN HISTORY ブロックとして再表示されるので、ベースラインは演習の最後まで画面に残ります。（1〜2分かかります。チケットを5回実行し、各試行が終わるたびに表示されます。）

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 3. モデルの問題か？
Priyaの仮説は、モデルの性能が足りないというものです。あなたも同じように考えているかもしれません。そこで正面から検証します。次のセルは、同じ評価をより大きく高価なモデルで実行します。新しい割合と新しいコストが表示されます。そのうえで、2つの問いに向き合ってください。大きなモデルは実際にチケットを解決しましたか？ それを確かめるのにいくらかかりましたか？ そこから何を持ち帰るにせよ、それはあなた自身が得た答えです。誰も教えてはいません。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5 --model claude-opus-4-8

## 4. チケットの処理を観察する
割合は失敗していることを教えてくれますが、どう失敗しているかは教えてくれません。そこで、1回の実行を最初から最後まで観察します。コーディネーターの動きがすべて表示されます。何を調べたか、どのスペシャリストにチケットを渡したか、お客様に送った返信。それをメールと照らし合わせて読んでください。お客様は複数のことを挙げていました。エージェントはそのうちいくつに対応し、どこで止まりましたか？

何を見ているのかわからない場合は、トランスクリプトをClaudeに貼り付けて、声に出して考えながら一緒に読み解いてください。*「これは、お客様が未解決だと言っているチケットをクローズしたサポートエージェントのトレースです。何をしたのか、どこで間違った可能性があるのかを順に説明してください。」* 返ってきた内容は、答えとして信じるのではなく、追いかけるべき手がかりとして扱ってください。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --ticket T-4471

## 5. エージェントの中に入る
失敗の仮説ができました。次は、それを生み出したシステムを見る番です。「エージェント」はコードではありません。このフォルダにあるただのファイルで、ここのエディタでそのまま開けます。

まずコーディネーターの `system-prompt-coordinator.txt` から始めます。各チケットを誰に担当させるかを決めているのはこのファイルなので、チケットの振り分けと引き継ぎがどう指示されているかを読んでください。その後は、ツール一覧（`coordinator-tools.json`）やスペシャリストのファイルも自由に見て構いません。

探すのは、さっき見たエージェントの振る舞いをまさに引き起こす指示や構成です。書き換えられるものを探してください。行き詰まったら、Claudeに聞いてみましょう。*「これはコーディネーターへの指示です。2つの別々の問題があるチケットで、片方しか処理されませんでした。この中に原因になりうる部分はありますか？」*

## 6. 1か所変えて、もう一度測る
何か見つかりましたか？ 変更し、ファイルを保存して、次のセルを実行します。割合は動きましたか？ スコアボードの下の RUN HISTORY ブロックには、ベースラインを含め、これまでのすべての実行が表示されるので、変更前後の比較はそこで確認できます。動かなかったなら、仮説が外れていたか、半分しか当たっていません。トレースに戻ってください。上がったなら、本物の調整項目を見つけたということです。エージェントが実行のたびにきれいにチケットを解決するようになるまで続けてください。最初から最後まで同じモデルのままです。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 7. 修正が持続することを確かめる
1つのチケットなら、偶然直ってしまうことがあります。次のセルは、ここまで見ていない別のチケット（ホールドアウト）を実行します。お客様も問題も別のものです。あなたの修正がこれらも解決するなら、本物です。解決しないなら、T-4471 に合わせて調整しただけで、まだやることが残っています。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --holdout --trials 3

## 8. Priyaに返信を書く
これで彼女に答えられます。`client-brief-template.md` に記入してください。実際に何がシステムを壊していたのかを平易な言葉で、何を変更したか、その証拠（変更前後の割合とコスト）、そして彼女が必ず聞いてくる問い、*「なぜもっと良いモデルにお金を払わないのですか？」* への答えです。その検証は手順3で行いました。答えはもうわかっています。